# Faces and Page Entries per Issue by Decade

This notebook summarizes the deduplicated face-detection dataset at the issue level.
It produces two outputs:

- a CSV with the fewest-face issues and the most-face issues, reported separately and combined for export: two per decade by default, except four most-face issues for the 1940s, restricted to issues published through 2007;
- an SVG figure with full-range decade summaries for detected faces per issue and page entries with faces per issue.

Execution convention:
Run this notebook from its own directory, `code/scripts`, so the fixed relative paths below resolve as intended.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 140
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
MAX_YEAR_CUTOFF = 2007
DEFAULT_OUTPUT_PER_DECADE = 4
SPECIAL_MOST_FACE_SELECTIONS = {1940: 4}

# Paths are relative to this notebook's directory: code/scripts.
deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
extreme_issues_csv = Path("../../data/processed/faces_per_issue_decade_extremes_through_2007.csv")
summary_figure_svg = Path("../../code/output/figures/faces_per_issue_decade_summary.svg")

assert deduplicated_csv.exists(), f"Missing input CSV: {deduplicated_csv}"
extreme_issues_csv.parent.mkdir(parents=True, exist_ok=True)
summary_figure_svg.parent.mkdir(parents=True, exist_ok=True)

expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

## Load and Validate the Deduplicated Face Dataset

The deduplicated CSV is the face-level input for this analysis. Each row is one retained face detection. The filename prefix identifies the issue and the source scan entry that produced the detection.

In [ ]:
faces = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})

assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated CSV is empty."

filename_parts = faces["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)
parse_failures = int(filename_parts["issue_id"].isna().sum())
assert parse_failures == 0, f"Could not parse {parse_failures:,} filenames."

faces = faces.assign(
    issue_id=filename_parts["issue_id"],
    source_pages=filename_parts["source_pages"],
    source_scan_id=(
        faces["Filename"]
        .str.replace(r"\.[^.]+$", "", regex=True)
        .str.split("_", n=1)
        .str[0]
    ),
)
faces["issue_date"] = pd.to_datetime(faces["issue_id"], format="%Y-%m%d")
faces["year"] = faces["issue_date"].dt.year
faces["decade"] = (faces["year"] // 10) * 10

pd.Series(
    {
        "face_rows": len(faces),
        "unique_source_scans": faces["source_scan_id"].nunique(),
        "unique_issues": faces["issue_id"].nunique(),
        "year_min": int(faces["year"].min()),
        "year_max": int(faces["year"].max()),
    },
    name="value",
).to_frame()

## Build Issue-Level Summaries

`detected_faces` counts deduplicated face detections per issue.

`pages_with_faces` counts unique `source_scan_id` values per issue. This treats a two-page scan entry such as `0045,0046` as one page entry:

In [ ]:
issue_summary = (
    faces.groupby("issue_id", as_index=False)
    .agg(
        issue_date=("issue_date", "first"),
        year=("year", "first"),
        decade=("decade", "first"),
        detected_faces=("Filename", "size"),
        pages_with_faces=("source_scan_id", "nunique"),
    )
    .sort_values("issue_date")
    .reset_index(drop=True)
)

assert issue_summary["issue_id"].is_unique
assert (issue_summary["detected_faces"] >= issue_summary["pages_with_faces"]).all()
assert issue_summary["year"].min() == faces["year"].min()
assert issue_summary["year"].max() == faces["year"].max()

issue_summary.head()

## Sanity Checks

These summaries document the issue-level scale of the dataset before selecting decade extremes.

In [ ]:
decade_overview = (
    issue_summary.groupby("decade")
    .agg(
        issues=("issue_id", "size"),
        median_detected_faces=("detected_faces", "median"),
        median_pages_with_faces=("pages_with_faces", "median"),
        max_detected_faces=("detected_faces", "max"),
        min_detected_faces=("detected_faces", "min"),
    )
    .reset_index()
)

decade_overview

## Select the Highest- and Lowest-Face Issues by Decade Through 2007

The decade selection uses only issues published on or before 2007. When several issues tie on `detected_faces`, the notebook uses seeded random tie-breaking with `RANDOM_SEED = 42` so the result is reproducible.

In [ ]:
issue_summary_through_2007 = issue_summary.loc[
    issue_summary["year"] <= MAX_YEAR_CUTOFF
].copy()

assert len(issue_summary_through_2007) > 0
assert int(issue_summary_through_2007["year"].max()) == MAX_YEAR_CUTOFF


def selections_per_decade(decade, mode):
    if mode == "most":
        return SPECIAL_MOST_FACE_SELECTIONS.get(int(decade), DEFAULT_OUTPUT_PER_DECADE)
    return DEFAULT_OUTPUT_PER_DECADE


def select_extreme_issues_by_decade(issue_df, mode, random_seed=RANDOM_SEED):
    if mode not in {"most", "least"}:
        raise ValueError('mode must be "most" or "least"')

    ascending = mode == "least"
    rng = random.Random(random_seed)
    selections = []

    for decade, decade_df in issue_df.groupby("decade", sort=True):
        per_decade = selections_per_decade(decade, mode)
        decade_df = decade_df.sort_values(["detected_faces", "issue_date", "issue_id"], ascending=[ascending, True, True])
        chosen_rows = []

        for face_count in sorted(decade_df["detected_faces"].unique(), reverse=not ascending):
            candidates = decade_df.loc[decade_df["detected_faces"] == face_count].copy()
            shuffled_positions = list(candidates.index)
            rng.shuffle(shuffled_positions)
            candidates = candidates.loc[shuffled_positions]
            remaining = per_decade - len(chosen_rows)
            chosen_rows.extend(candidates.head(remaining).to_dict("records"))
            if len(chosen_rows) == per_decade:
                break

        for sample_order, row in enumerate(chosen_rows, start=1):
            selections.append(
                {
                    "decade": decade,
                    "mode": mode,
                    "sample_order": sample_order,
                    "issue_id": row["issue_id"],
                    "issue_date": row["issue_date"].date().isoformat(),
                    "year": int(row["year"]),
                    "detected_faces": int(row["detected_faces"]),
                    "pages_with_faces": int(row["pages_with_faces"]),
                }
            )

    return pd.DataFrame(selections)


highest_face_issues = select_extreme_issues_by_decade(issue_summary_through_2007, mode="most")
lowest_face_issues = select_extreme_issues_by_decade(issue_summary_through_2007, mode="least")

decade_extremes = (
    pd.concat([highest_face_issues, lowest_face_issues], ignore_index=True)
    .sort_values(["decade", "mode", "sample_order"], ascending=[True, False, True])
    .reset_index(drop=True)
)

expected_rows = sum(
    selections_per_decade(decade, "most") + selections_per_decade(decade, "least")
    for decade in sorted(decade_extremes["decade"].unique())
)
assert len(decade_extremes) == expected_rows

with pd.option_context("display.max_rows", None):
    print("Most-face issues by decade")
    display(highest_face_issues)
    print("Fewest-face issues by decade")
    display(lowest_face_issues)
    print("Combined export table")
    display(decade_extremes)

## Write and Verify the Decade-Extremes Output

The CSV keeps one row per sampled issue so it can be reused directly in later analysis or in thesis tables.

In [ ]:
decade_extremes.to_csv(extreme_issues_csv, index=False)

reloaded_extremes = pd.read_csv(extreme_issues_csv)
assert len(reloaded_extremes) == len(decade_extremes)
assert list(reloaded_extremes.columns) == list(decade_extremes.columns)

reloaded_extremes.head()

## Plot Full-Range Decade Summaries

The final figure uses the full date range of the deduplicated dataset, not the 2007 cutoff. It summarizes the distribution of issue-level counts by decade for both requested metrics.

In [ ]:
#TODO: decide, if needed
plot_summary = issue_summary.copy()
plot_summary["decade_label"] = plot_summary["decade"].astype(str) + "s"
plot_summary = plot_summary.sort_values(["decade", "issue_date"]).reset_index(drop=True)

decade_order = plot_summary["decade_label"].drop_duplicates().tolist()

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 9), sharey=True, constrained_layout=True)

metric_specs = [
    ("detected_faces", "Detected faces per issue", "#4c78a8"),
    ("pages_with_faces", "Page entries with faces per issue", "#f58518"),
]

for ax, (column, title, color) in zip(axes, metric_specs):
    sns.boxplot(
        data=plot_summary,
        y="decade_label",
        x=column,
        order=decade_order,
        ax=ax,
        color=color,
        showfliers=False,
        linewidth=1,
    )
    sns.stripplot(
        data=plot_summary,
        y="decade_label",
        x=column,
        order=decade_order,
        ax=ax,
        color="#222222",
        alpha=0.18,
        size=2,
        jitter=0.22,
    )
    ax.set_title(title)
    ax.set_xlabel("Issues")
    ax.set_ylabel("Decade")

fig.suptitle("Issue-level face and page-entry distributions by decade", fontsize=14)
fig.savefig(summary_figure_svg, format="svg", bbox_inches="tight")
plt.show()

summary_figure_svg